# SSSOM Mapping Analysis

Analysis of SSSOM (Simple Standard for Sharing Ontological Mappings) files generated by RDFSolve.

In [ ]:
from collections import Counter
from pathlib import Path

import pandas as pd

OUTPUT_DIR = Path("/home/javier.millanacosta/rdfsolve/output")
MAPPINGS_DIR = OUTPUT_DIR / "mappings"
print(f"Mappings directory: {MAPPINGS_DIR}")

In [ ]:
def parse_sssom_tsv(filepath: Path) -> tuple[dict, list[dict]]:
    """Parse SSSOM TSV file returning metadata and mappings."""
    metadata = {}
    mappings = []
    headers = []
    
    with filepath.open() as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith("#"):
                content = line[1:].strip()
                if ":" in content:
                    key, _, value = content.partition(":")
                    metadata[key.strip()] = value.strip()
            elif not headers:
                headers = line.split("\t")
            else:
                values = line.split("\t")
                mappings.append(dict(zip(headers, values)))
    
    return metadata, mappings

In [ ]:
# Load all SSSOM files
sssom_files = list(MAPPINGS_DIR.glob("**/*.sssom.tsv"))
print(f"Found {len(sssom_files)} SSSOM files")

all_mappings = []
file_stats = []

for f in sssom_files:
    metadata, mappings = parse_sssom_tsv(f)
    file_stats.append({
        "file": f.name,
        "path": str(f.relative_to(MAPPINGS_DIR)),
        "mappings": len(mappings),
        "subject_source": metadata.get("subject_source", ""),
        "object_source": metadata.get("object_source", ""),
    })
    for m in mappings:
        m["_file"] = f.name
        all_mappings.append(m)

df_files = pd.DataFrame(file_stats).sort_values("mappings", ascending=False)
print(f"Total mappings: {len(all_mappings):,}")
df_files

In [ ]:
# Predicate distribution
if all_mappings:
    predicate_counter = Counter(m.get("predicate_id", "") for m in all_mappings)
    print("Predicates used:")
    for pred, count in predicate_counter.most_common():
        pct = count / len(all_mappings) * 100
        print(f"  {count:6d} ({pct:5.1f}%)  {pred}")

In [ ]:
# Mapping justification distribution
if all_mappings:
    just_counter = Counter(m.get("mapping_justification", "") for m in all_mappings)
    print("Mapping justifications:")
    for just, count in just_counter.most_common():
        pct = count / len(all_mappings) * 100
        print(f"  {count:6d} ({pct:5.1f}%)  {just}")

In [ ]:
# Confidence distribution
if all_mappings:
    confidences = []
    for m in all_mappings:
        conf = m.get("confidence", "")
        if conf:
            try:
                confidences.append(float(conf))
            except ValueError:
                pass
    
    if confidences:
        print(f"Confidence statistics (n={len(confidences)}):")
        print(f"  Min: {min(confidences):.2f}")
        print(f"  Max: {max(confidences):.2f}")
        print(f"  Avg: {sum(confidences) / len(confidences):.2f}")

In [ ]:
# Namespace distribution in mappings
def extract_namespace(uri: str) -> str:
    if "#" in uri:
        return uri.rsplit("#", 1)[0] + "#"
    elif "/" in uri:
        return uri.rsplit("/", 1)[0] + "/"
    return uri

if all_mappings:
    subj_ns = Counter(extract_namespace(m.get("subject_id", "")) for m in all_mappings if m.get("subject_id"))
    obj_ns = Counter(extract_namespace(m.get("object_id", "")) for m in all_mappings if m.get("object_id"))
    
    print("Top 10 subject namespaces:")
    for ns, count in subj_ns.most_common(10):
        print(f"  {count:5d}  {ns[:60]}")
    
    print("\nTop 10 object namespaces:")
    for ns, count in obj_ns.most_common(10):
        print(f"  {count:5d}  {ns[:60]}")

In [ ]:
# Check enriched external mappings
enriched_dir = MAPPINGS_DIR / "enriched"
if enriched_dir.exists():
    enriched_files = list(enriched_dir.glob("*.sssom.tsv"))
    print(f"Enriched SSSOM files: {len(enriched_files)}")
    for f in enriched_files:
        _, mappings = parse_sssom_tsv(f)
        print(f"  - {f.name}: {len(mappings)} mappings")
else:
    print("No enriched mappings directory found")